Transfer Learning with VGG16, ResNet50, and VGG+ResNet Ensemble

**Disease Classes:**
1. CNV
2. DME
3. DRUSEN
4. NORMAL

**Models:**
- VGG16 (Transfer Learning)
- ResNet50 (Transfer Learning)
- VGG16 + ResNet50 Ensemble (Combined)

**Evaluation Metrics:** Accuracy, Precision, Recall, F1-Score, AUC-ROC, Confusion Matrix

## 1. Install & Import Libraries

In [1]:
# Install required libraries (run once)
!pip install tensorflow scikit-learn matplotlib seaborn numpy pandas opencv-python-headless kaggle -q

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
import warnings
warnings.filterwarnings('ignore')
    
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, auc
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

TensorFlow version: 2.21.0
GPU Available: False


## 2. Configuration

In [10]:
# ─────────────────────────────────────────────
# CONFIGURATION — Edit these paths as needed
# ─────────────────────────────────────────────

DATA_DIR = r'D:\rscbjbr9sj-3\ZhangLabData\CellData\OCT\train'          # Change to your dataset directory

# Image settings
IMG_HEIGHT = 224
IMG_WIDTH  = 224
IMG_SIZE   = (IMG_HEIGHT, IMG_WIDTH)
CHANNELS   = 3
INPUT_SHAPE = (IMG_HEIGHT, IMG_WIDTH, CHANNELS)

# Training settings
BATCH_SIZE  = 32
EPOCHS      = 1
LEARNING_RATE = 1e-4
TEST_SIZE   = 0.2
VAL_SIZE    = 0.1

# Class names
CLASS_NAMES = ['CNV', 'DME', 'DRUSEN', 'NORMAL'] 
NUM_CLASSES = len(CLASS_NAMES)

print("Configuration set:")
print(f"  Image size   : {IMG_SIZE}")
print(f"  Batch size   : {BATCH_SIZE}")
print(f"  Epochs       : {EPOCHS}")
print(f"  Num classes  : {NUM_CLASSES}")
print(f"  Classes      : {CLASS_NAMES}")

Configuration set:
  Image size   : (224, 224)
  Batch size   : 32
  Epochs       : 1
  Num classes  : 4
  Classes      : ['CNV', 'DME', 'DRUSEN', 'NORMAL']


## 4. Data Loading & Preprocessing

In [ ]:
def load_and_preprocess_images(data_dir, class_names, img_size):
    images, labels = [], []
    label_map = {name: idx for idx, name in enumerate(class_names)}

    for class_name in class_names:
        class_dir = os.path.join(data_dir, class_name)
        if not os.path.exists(class_dir):
            print(f"Directory not found: {class_dir}")
            continue

        files = [f for f in os.listdir(class_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        print(f"  Loading {len(files):>5} images for: {class_name}")

        for fname in files:
            path = os.path.join(class_dir, fname)
            img = cv2.imread(path, cv2.IMREAD_COLOR)
            if img is None:
                print(f"Failed to read: {path}")
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, img_size)
            img = img.astype(np.float32) / 255.0
            images.append(img)
            labels.append(label_map[class_name])

    return np.array(images), np.array(labels)

Loading dataset (limiting to 200 images per class)...
  Loading   200 images for: CNV
  Loading   200 images for: DME
  Loading   200 images for: DRUSEN


: 

## 5. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution bar chart
counts = np.bincount(y)
colors = ['#2ECC71', '#E74C3C', '#3498DB', '#F39C12']
bars = axes[0].bar(CLASS_NAMES, counts, color=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Disease Class')
axes[0].set_ylabel('Number of Images')
axes[0].tick_params(axis='x', rotation=15)
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 str(count), ha='center', va='bottom', fontweight='bold')

# Pie chart
axes[1].pie(counts, labels=CLASS_NAMES, colors=colors, autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Class Distribution (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Total images: {len(X)} | Classes: {NUM_CLASSES}")

In [ ]:
# Sample images from each class
fig, axes = plt.subplots(NUM_CLASSES, 5, figsize=(15, NUM_CLASSES * 3))
fig.suptitle('Sample Images per Disease Class', fontsize=16, fontweight='bold', y=1.01)

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    cls_indices = np.where(y == cls_idx)[0]
    sample_idx  = np.random.choice(cls_indices, min(5, len(cls_indices)), replace=False)
    for col, idx in enumerate(sample_idx):
        ax = axes[cls_idx][col]
        ax.imshow(X[idx])
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(cls_name, fontsize=11, fontweight='bold', rotation=90,
                          labelpad=10, va='center')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Train / Validation / Test Split

In [ ]:
# One-hot encode labels
y_cat = to_categorical(y, NUM_CLASSES)

# Train+Val / Test split
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y_cat, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

# Train / Val split
val_frac = VAL_SIZE / (1 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=val_frac, random_state=SEED,
    stratify=np.argmax(y_trainval, axis=1)
)

print("Data splits:")
print(f"  Training   : {len(X_train):>5} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Validation : {len(X_val):>5} samples ({len(X_val)/len(X)*100:.1f}%)")
print(f"  Test       : {len(X_test):>5} samples ({len(X_test)/len(X)*100:.1f}%)")

## 7. Data Augmentation

In [ ]:
# Training augmentation pipeline
train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

val_datagen  = ImageDataGenerator()   # No augmentation for val/test
test_datagen = ImageDataGenerator()

train_generator = train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE, seed=SEED)
val_generator   = val_datagen.flow(X_val,   y_val,   batch_size=BATCH_SIZE, shuffle=False)
test_generator  = test_datagen.flow(X_test,  y_test,  batch_size=BATCH_SIZE, shuffle=False)

# Visualise augmented samples
sample_img = X_train[:1]
sample_lbl = y_train[:1]
aug_gen    = train_datagen.flow(sample_img, sample_lbl, batch_size=1)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Original (top-left) vs Augmented Samples', fontsize=13, fontweight='bold')
axes[0][0].imshow(sample_img[0]); axes[0][0].set_title('Original'); axes[0][0].axis('off')
for i, ax in enumerate(axes.flat[1:]):
    aug_img, _ = next(aug_gen)
    ax.imshow(np.clip(aug_img[0], 0, 1))
    ax.set_title(f'Aug {i+1}'); ax.axis('off')
plt.tight_layout()
plt.savefig('augmentation_samples.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Model Building

### 8.1 Utility Functions

In [ ]:
def get_callbacks(model_name):
    """Standard callbacks for training."""
    return [
        EarlyStopping(
            monitor='val_loss', patience=7, restore_best_weights=True, verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1
        ),
        ModelCheckpoint(
            filepath=f'best_{model_name}.keras',
            monitor='val_accuracy', save_best_only=True, verbose=0
        )
    ]


def compile_model(model, lr=LEARNING_RATE):
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy',
                 keras.metrics.AUC(name='auc'),
                 keras.metrics.Precision(name='precision'),
                 keras.metrics.Recall(name='recall')]
    )
    return model


def classification_head(base_output, name_prefix):
    """Standard dense head for transfer learning models."""
    x = layers.GlobalAveragePooling2D(name=f'{name_prefix}_gap')(base_output)
    x = layers.BatchNormalization(name=f'{name_prefix}_bn1')(x)
    x = layers.Dense(512, activation='relu', name=f'{name_prefix}_fc1')(x)
    x = layers.Dropout(0.4, name=f'{name_prefix}_drop1')(x)
    x = layers.Dense(256, activation='relu', name=f'{name_prefix}_fc2')(x)
    x = layers.Dropout(0.3, name=f'{name_prefix}_drop2')(x)
    out = layers.Dense(NUM_CLASSES, activation='softmax', name=f'{name_prefix}_out')(x)
    return out


print("Utility functions defined.")

### 8.2 VGG16 Transfer Learning Model

In [ ]:
def build_vgg_model(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES, fine_tune_at=15):
    """
    VGG16 transfer learning model.
    - Loads ImageNet weights
    - Freezes early layers, fine-tunes from `fine_tune_at`
    """
    base = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)

    # Freeze all then unfreeze from fine_tune_at
    base.trainable = True
    for layer in base.layers[:fine_tune_at]:
        layer.trainable = False

    inp  = keras.Input(shape=input_shape, name='vgg_input')
    x    = base(inp, training=False)
    out  = classification_head(x, 'vgg')
    model = Model(inputs=inp, outputs=out, name='VGG16_Transfer')
    return model


vgg_model = build_vgg_model()
vgg_model = compile_model(vgg_model)
vgg_model.summary()

### 8.3 ResNet50 Transfer Learning Model

In [ ]:
def build_resnet_model(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES, fine_tune_at=140):
    """
    ResNet50 transfer learning model.
    - Loads ImageNet weights
    - Freezes early layers, fine-tunes from `fine_tune_at`
    """
    base = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)

    base.trainable = True
    for layer in base.layers[:fine_tune_at]:
        layer.trainable = False

    inp  = keras.Input(shape=input_shape, name='resnet_input')
    x    = base(inp, training=False)
    out  = classification_head(x, 'resnet')
    model = Model(inputs=inp, outputs=out, name='ResNet50_Transfer')
    return model


resnet_model = build_resnet_model()
resnet_model = compile_model(resnet_model)
resnet_model.summary()

### 8.4 VGG16 + ResNet50 Ensemble Model

In [ ]:
def build_ensemble_model(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES):
    """
    Ensemble of VGG16 and ResNet50 feature extractors.
    Features from both backbones are concatenated before the final head.
    """
    inp = keras.Input(shape=input_shape, name='ensemble_input')

    # ── VGG16 branch ──
    vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)
    vgg_base.trainable = True
    for layer in vgg_base.layers[:15]:
        layer.trainable = False
    vgg_base._name = 'vgg16_backbone'

    vgg_feat = vgg_base(inp, training=False)
    vgg_feat = layers.GlobalAveragePooling2D(name='vgg_gap')(vgg_feat)
    vgg_feat = layers.Dense(256, activation='relu', name='vgg_proj')(vgg_feat)
    vgg_feat = layers.Dropout(0.3, name='vgg_drop')(vgg_feat)

    # ── ResNet50 branch ──
    rn_base = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    rn_base.trainable = True
    for layer in rn_base.layers[:140]:
        layer.trainable = False
    rn_base._name = 'resnet50_backbone'

    rn_feat = rn_base(inp, training=False)
    rn_feat = layers.GlobalAveragePooling2D(name='rn_gap')(rn_feat)
    rn_feat = layers.Dense(256, activation='relu', name='rn_proj')(rn_feat)
    rn_feat = layers.Dropout(0.3, name='rn_drop')(rn_feat)

    # ── Fusion head ──
    merged = layers.Concatenate(name='fusion')([vgg_feat, rn_feat])
    merged = layers.BatchNormalization(name='fusion_bn')(merged)
    merged = layers.Dense(256, activation='relu', name='fusion_fc1')(merged)
    merged = layers.Dropout(0.4, name='fusion_drop')(merged)
    output = layers.Dense(num_classes, activation='softmax', name='fusion_out')(merged)

    model = Model(inputs=inp, outputs=output, name='VGG_ResNet_Ensemble')
    return model


ensemble_model = build_ensemble_model()
ensemble_model = compile_model(ensemble_model)
ensemble_model.summary()

## 9. Training

In [ ]:
def plot_training_history(history, model_name):
    """Plot loss, accuracy, and AUC curves."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'{model_name} — Training History', fontsize=14, fontweight='bold')

    metrics = [('loss', 'Loss'), ('accuracy', 'Accuracy'), ('auc', 'AUC')]
    colors  = [('#E74C3C', '#C0392B'), ('#2ECC71', '#27AE60'), ('#3498DB', '#2980B9')]

    for ax, (metric, title), (tc, vc) in zip(axes, metrics, colors):
        if metric in history.history:
            ax.plot(history.history[metric],          color=tc, linewidth=2, label='Train')
            ax.plot(history.history[f'val_{metric}'], color=vc, linewidth=2,
                    linestyle='--', label='Validation')
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel(title)
        ax.legend()
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'training_{model_name.lower().replace(" ","_")}.png', dpi=150, bbox_inches='tight')
    plt.show()

print("Training function defined.")

In [ ]:
# ── Train VGG16 ──
print("="*50)
print(" Training VGG16 Transfer Learning Model")
print("="*50)

vgg_history = vgg_model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=get_callbacks('vgg16'),
    verbose=1
)

plot_training_history(vgg_history, 'VGG16')

In [ ]:
# ── Train ResNet50 ──
print("="*50)
print(" Training ResNet50 Transfer Learning Model")
print("="*50)

resnet_history = resnet_model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=get_callbacks('resnet50'),
    verbose=1
)

plot_training_history(resnet_history, 'ResNet50')

In [ ]:
# ── Train Ensemble ──
print("="*50)
print(" Training VGG16 + ResNet50 Ensemble Model")
print("="*50)

ensemble_history = ensemble_model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=get_callbacks('ensemble'),
    verbose=1
)

plot_training_history(ensemble_history, 'VGG_ResNet_Ensemble')

## 10. Evaluation

In [ ]:
def evaluate_model(model, X_test, y_test_cat, class_names, model_name):
    """
    Full evaluation pipeline:
    - Accuracy, Precision, Recall, F1 (macro & weighted)
    - Cohen's Kappa
    - AUC-ROC (macro OvR)
    - Confusion matrix
    - Per-class classification report
    - ROC curves
    """
    from sklearn.metrics import cohen_kappa_score

    y_true  = np.argmax(y_test_cat, axis=1)
    y_prob  = model.predict(X_test, verbose=0)
    y_pred  = np.argmax(y_prob, axis=1)

    # ── Scalar metrics ──
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    f1m  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    kappa = cohen_kappa_score(y_true, y_pred)

    try:
        auc_roc = roc_auc_score(y_test_cat, y_prob, multi_class='ovr', average='macro')
    except Exception:
        auc_roc = float('nan')

    results = {
        'Model':             model_name,
        'Accuracy':          acc,
        'Precision (W)':     prec,
        'Recall (W)':        rec,
        'F1 Weighted':       f1,
        'F1 Macro':          f1m,
        "Cohen's Kappa":     kappa,
        'AUC-ROC (macro)':   auc_roc
    }

    print(f"\n{'='*55}")
    print(f"  {model_name} — Test Set Evaluation")
    print(f"{'='*55}")
    for k, v in results.items():
        if k != 'Model':
            print(f"  {k:<22}: {v:.4f}")

    print(f"\n  Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

    return results, y_true, y_pred, y_prob


print("Evaluation function defined.")

In [ ]:
# Run evaluation for all models
vgg_results,    y_true, vgg_pred,    vgg_prob    = evaluate_model(vgg_model,      X_test, y_test, CLASS_NAMES, 'VGG16')
resnet_results, _,      resnet_pred, resnet_prob  = evaluate_model(resnet_model,   X_test, y_test, CLASS_NAMES, 'ResNet50')
ens_results,    _,      ens_pred,    ens_prob     = evaluate_model(ensemble_model, X_test, y_test, CLASS_NAMES, 'VGG+ResNet Ensemble')

## 11. Confusion Matrices

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names, model_name, ax, normalize=True):
    cm = confusion_matrix(y_true, y_pred)
    if normalize:
        cm_plot = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        fmt, vmax = '.2f', 1.0
    else:
        cm_plot, fmt, vmax = cm, 'd', cm.max()

    sns.heatmap(
        cm_plot, annot=True, fmt=fmt, cmap='Blues',
        xticklabels=class_names, yticklabels=class_names,
        ax=ax, linewidths=0.5, linecolor='gray',
        vmin=0, vmax=vmax, cbar_kws={'shrink': 0.8}
    )
    ax.set_title(f'{model_name}\n(Normalized)' if normalize else model_name,
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('True',      fontsize=10)
    ax.tick_params(axis='x', rotation=30)
    ax.tick_params(axis='y', rotation=0)


fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Confusion Matrices — All Models', fontsize=15, fontweight='bold')

plot_confusion_matrix(y_true, vgg_pred,    CLASS_NAMES, 'VGG16',               axes[0])
plot_confusion_matrix(y_true, resnet_pred, CLASS_NAMES, 'ResNet50',             axes[1])
plot_confusion_matrix(y_true, ens_pred,    CLASS_NAMES, 'VGG + ResNet Ensemble',axes[2])

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. ROC Curves

In [ ]:
def plot_roc_curves(y_true_cat, y_prob, class_names, model_name, ax):
    colors = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12']
    for i, (cls_name, color) in enumerate(zip(class_names, colors)):
        fpr, tpr, _ = roc_curve(y_true_cat[:, i], y_prob[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, lw=2, label=f'{cls_name} (AUC={roc_auc:.3f})')

    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'{model_name}\nROC Curves', fontsize=12, fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(alpha=0.3)


fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('ROC Curves — All Models', fontsize=15, fontweight='bold')

plot_roc_curves(y_test, vgg_prob,    CLASS_NAMES, 'VGG16',               axes[0])
plot_roc_curves(y_test, resnet_prob, CLASS_NAMES, 'ResNet50',             axes[1])
plot_roc_curves(y_test, ens_prob,    CLASS_NAMES, 'VGG + ResNet Ensemble',axes[2])

plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Model Comparison

In [ ]:
# Build comparison DataFrame
comparison_df = pd.DataFrame([vgg_results, resnet_results, ens_results])
comparison_df = comparison_df.set_index('Model').round(4)

print("\n" + "="*70)
print("  MODEL COMPARISON TABLE")
print("="*70)
print(comparison_df.to_string())
print("="*70)

# Highlight best performer
best_acc_model = comparison_df['Accuracy'].idxmax()
best_auc_model = comparison_df['AUC-ROC (macro)'].idxmax()
print(f"\n  Best Accuracy : {best_acc_model} ({comparison_df.loc[best_acc_model,'Accuracy']:.4f})")
print(f"  Best AUC-ROC  : {best_auc_model} ({comparison_df.loc[best_auc_model,'AUC-ROC (macro)']:.4f})")

In [ ]:
# Visual comparison bar charts
metrics_to_plot = ['Accuracy', 'Precision (W)', 'Recall (W)', 'F1 Weighted', 'AUC-ROC (macro)']
models = comparison_df.index.tolist()
x = np.arange(len(metrics_to_plot))
width = 0.25
colors_m = ['#3498DB', '#E74C3C', '#2ECC71']

fig, ax = plt.subplots(figsize=(14, 6))
for i, (model_name, color) in enumerate(zip(models, colors_m)):
    vals = [comparison_df.loc[model_name, m] for m in metrics_to_plot]
    bars = ax.bar(x + i*width, vals, width, label=model_name, color=color,
                  edgecolor='black', linewidth=0.6, alpha=0.9)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8, rotation=60)

ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim([0, 1.12])
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Per-Class F1 Heatmap

In [ ]:
from sklearn.metrics import classification_report
import json

def per_class_f1(y_true, y_pred, class_names):
    report = classification_report(y_true, y_pred, target_names=class_names,
                                   output_dict=True, zero_division=0)
    return {cls: report[cls]['f1-score'] for cls in class_names}


f1_vgg    = per_class_f1(y_true, vgg_pred,    CLASS_NAMES)
f1_resnet = per_class_f1(y_true, resnet_pred, CLASS_NAMES)
f1_ens    = per_class_f1(y_true, ens_pred,    CLASS_NAMES)

f1_df = pd.DataFrame(
    [f1_vgg, f1_resnet, f1_ens],
    index=['VGG16', 'ResNet50', 'Ensemble']
)

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(f1_df, annot=True, fmt='.3f', cmap='YlOrRd',
            linewidths=0.5, vmin=0, vmax=1, ax=ax,
            cbar_kws={'label': 'F1 Score'})
ax.set_title('Per-Class F1 Score Heatmap', fontsize=14, fontweight='bold')
ax.set_xlabel('Disease Class')
ax.set_ylabel('Model')
plt.tight_layout()
plt.savefig('f1_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 15. Misclassified Samples

In [ ]:
def plot_misclassified(X_test, y_true, y_pred, class_names, model_name, n=10):
    wrong_idx = np.where(y_true != y_pred)[0]
    if len(wrong_idx) == 0:
        print(f"{model_name}: No misclassified samples!")
        return

    n = min(n, len(wrong_idx))
    sample = np.random.choice(wrong_idx, n, replace=False)

    cols = 5
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*3, rows*3))
    fig.suptitle(f'{model_name} — Misclassified Examples\n'
                 f'({len(wrong_idx)} total errors)', fontsize=13, fontweight='bold')

    for ax, idx in zip(axes.flat, sample):
        ax.imshow(X_test[idx])
        ax.set_title(f'True: {class_names[y_true[idx]]}\nPred: {class_names[y_pred[idx]]}',
                     fontsize=9, color='red')
        ax.axis('off')

    for ax in axes.flat[n:]:
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(f'misclassified_{model_name.lower().replace(" ","_")}.png',
                dpi=120, bbox_inches='tight')
    plt.show()


plot_misclassified(X_test, y_true, vgg_pred,    CLASS_NAMES, 'VGG16')
plot_misclassified(X_test, y_true, resnet_pred, CLASS_NAMES, 'ResNet50')
plot_misclassified(X_test, y_true, ens_pred,    CLASS_NAMES, 'VGG+ResNet Ensemble')

## 16. Save Models & Results

In [ ]:
# Save trained models
vgg_model.save('vgg16_eye_disease.keras')
resnet_model.save('resnet50_eye_disease.keras')
ensemble_model.save('ensemble_eye_disease.keras')

# Save metrics table
comparison_df.to_csv('model_comparison_results.csv')

print("✅ Models saved:")
print("   vgg16_eye_disease.keras")
print("   resnet50_eye_disease.keras")
print("   ensemble_eye_disease.keras")
print("\n✅ Results saved:")
print("   model_comparison_results.csv")
print("\n✅ Plots saved:")
for fname in ['class_distribution', 'sample_images', 'augmentation_samples',
              'training_vgg16', 'training_resnet50', 'training_vgg_resnet_ensemble',
              'confusion_matrices', 'roc_curves', 'model_comparison', 'f1_heatmap']:
    print(f"   {fname}.png")

## 17. Inference — Single Image Prediction

In [ ]:
def predict_single_image(image_path, model, class_names, img_size=IMG_SIZE):
    """
    Predict eye disease class for a single image file.
    Returns predicted class and confidence scores.
    """
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Cannot read image: {image_path}")

    img_rgb   = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resz  = cv2.resize(img_rgb, img_size)
    img_norm  = img_resz.astype(np.float32) / 255.0
    img_batch = np.expand_dims(img_norm, axis=0)

    probs   = model.predict(img_batch, verbose=0)[0]
    cls_idx = np.argmax(probs)

    # Visualise
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(img_norm)
    axes[0].set_title(f'Predicted: {class_names[cls_idx]}\n'
                      f'Confidence: {probs[cls_idx]*100:.1f}%', fontsize=12, fontweight='bold')
    axes[0].axis('off')

    colors = ['#2ECC71' if i == cls_idx else '#95A5A6' for i in range(len(class_names))]
    axes[1].barh(class_names, probs, color=colors, edgecolor='black', linewidth=0.5)
    axes[1].set_xlabel('Probability')
    axes[1].set_title('Class Probabilities')
    axes[1].set_xlim([0, 1])
    for i, (p, bar) in enumerate(zip(probs, axes[1].patches)):
        axes[1].text(min(p + 0.02, 0.95), bar.get_y() + bar.get_height()/2,
                     f'{p*100:.1f}%', va='center', fontsize=10)

    plt.tight_layout()
    plt.show()

    return class_names[cls_idx], probs


# Example usage:
# cls, probs = predict_single_image('./test_eye.jpg', ensemble_model, CLASS_NAMES)
print("Inference function defined.")
print("Usage: predict_single_image('./path/to/eye.jpg', ensemble_model, CLASS_NAMES)")

## 18. Final Summary

In [ ]:
print("\n" + "━"*60)
print("  EYE DISEASE CLASSIFICATION — FINAL RESULTS SUMMARY")
print("━"*60)
print(comparison_df[['Accuracy','F1 Weighted','AUC-ROC (macro)']].\
      sort_values('Accuracy', ascending=False).to_string())
print("━"*60)

best = comparison_df['Accuracy'].idxmax()
print(f"\n  🏆 Best model (Accuracy): {best}")
print(f"     Accuracy  : {comparison_df.loc[best,'Accuracy']:.4f}")
print(f"     F1 Score  : {comparison_df.loc[best,'F1 Weighted']:.4f}")
print(f"     AUC-ROC   : {comparison_df.loc[best,'AUC-ROC (macro)']:.4f}")
print("\n  📁 Generated files:")
print("     Models     : *_eye_disease.keras")
print("     Metrics    : model_comparison_results.csv")
print("     Plots      : *.png")